# Curriculum 05 · Lab 1 — Query rewrite: fix the query before you embed it

**Goal:** Fix the retrieval bottleneck that costs nothing but one cheap LLM
call: a vague or conversational question ("did he win it?", "what about the
second one?") embeds into the wrong region of vector space, so the top-k
passages miss before any answer LLM runs. Query rewrite turns the user's
question into a **standalone, specific search query first**, and the
retriever embeds *that*.

```
Transformer : QueryRewriteRetriever (retrieval/query_rewrite.py)
LLM         : llama-3.3-70b-versatile (GroqLLM) — the rewriter, never the embedder
Store       : FAISSVectorStore (in-memory) + SimilarityRetriever
Embedding   : BGE (BAAI/bge-base-en-v1.5, local, CPU)
Data        : rag-mini-wikipedia — first 100 passages, questions 1606/1610/1626
```

**Why rewrite first:** retrieval quality is capped by query quality. The
technique is a retriever-layer swap — the store and embedding model never
change, only the text that reaches them. Cost: one LLM call per question.
Payoff: pronouns, ellipses and conversational phrasing stop leaking into the
embedding.

This is the first lab of track 05-query-transformation (see
`.omo/plans/layer1-rag-playbook.md`).


## 0 · Setup — environment, imports & repo paths

**WHAT:** Installs the lab's dependencies (a no-op if already present),
loads `GROQ_API_KEY` from the repo-root `.env`, and puts the repo-root
component library on `sys.path` so this notebook reuses `src/retrieval/*.py`,
`src/llms/groq.py`, `src/vectordb/faiss.py` and `src/embeddings/bge.py` exactly like the
lab script.

**WHY:** Everything embeds **locally** with BGE via sentence-transformers —
no API embeddings anywhere. The LLM is only the query-*transformation* step
(Groq's `llama-3.3-70b-versatile`; a commented Gemini alternative is kept in
the source). The retriever classes live in the repo's shared component
library (`src/retrieval/`), not inside the lab, so the exact same code path runs
here, in the `.py`, and in later tracks.

**Paths:** the next cell resolves the **repo root** automatically — it works
whether the kernel launches from the repo root (like the lab script) or from
the notebook's own folder (the Jupyter default) — and `cd`s into it so every
path stays repo-relative.

**WHAT TO EXPECT:** no output from the pip cell (packages already
installed), a silent import from the second. The BGE model is loaded lazily
when the experiment cell first calls it; the Groq key is read from `.env`.


In [1]:
# Lab-specific dependencies (already in requirements.txt — the install is a
# no-op safety net for fresh environments):
#   sentence-transformers -> local BGE embeddings (embeddings/bge.py)
#   faiss-cpu             -> the FAISS index (vectordb/faiss.py)
#   langchain-groq        -> GroqLLM (the rewriter LLM, llms/groq.py)
#   python-dotenv         -> loads GROQ_API_KEY from the repo-root .env
#   pandas                -> reads the passages/test.parquet corpus
%pip install sentence-transformers faiss-cpu langchain-groq python-dotenv pandas



[notice] A new release of pip is available: 24.2 -> 26.2
[notice] To update, run: python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import os
import sys
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

# Make the repo-root component library importable. A notebook has no
# ``__file__``, so resolve the repo root by walking up from the kernel's
# working directory — this works whether the kernel launches from the repo
# root (like the lab script) or from the notebook's own folder (Jupyter's
# default) — then cd into it so every repo-relative path behaves exactly
# like the .py.
REPO_ROOT = Path.cwd()
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "src" / "curriculum").is_dir() and (candidate / "NoteBooks").is_dir():
        REPO_ROOT = candidate
        break
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))

load_dotenv(REPO_ROOT / ".env")  # GROQ_API_KEY lives in the repo-root .env

from embeddings.bge import BGEEmbedding  # noqa: E402
from langchain_core.documents import Document  # noqa: E402
from llms.groq import GroqLLM  # noqa: E402
from retrieval.query_rewrite import QueryRewriteRetriever  # noqa: E402
from retrieval.similarity import SimilarityRetriever  # noqa: E402
from vectordb.faiss import FAISSVectorStore  # noqa: E402


## 1 · Configuration — the experiment's knobs

**WHAT:** The corpus constants (`N_PASSAGES = 100`, `QUESTION_IDS =
[1606, 1610, 1626]`, `TOP_K = 3`) plus `LLM_MODEL =
"llama-3.3-70b-versatile"` — the Groq model that rewrites the questions (a
commented Gemini alternative is kept in the source).

**WHY:** The three questions are deliberately conversational/yes-no so the
rewrite has real work to do ("Is Uruguay's capital Montevideo?" ->
"Which city is the capital of Uruguay?"), and their gold answers live inside
the 100-passage subset, so the gate can check the rewritten path still finds
the answer.


In [3]:
PASSAGES_PATH = Path("Data/corpus/rag-mini-wikipedia/passages.parquet")
TEST_PATH = Path("Data/corpus/rag-mini-wikipedia/test.parquet")
N_PASSAGES = 100  # deterministic head of the 3200-passage corpus (keeps runtime low)
# Vague/conversational questions whose gold answers live in the subset; the
# rewrite must resolve them into standalone queries that still find the answer.
QUESTION_IDS = [1606, 1610, 1626]
TOP_K = 3
LLM_MODEL = "llama-3.3-70b-versatile"  # Groq is the *rewriter* LLM, never the embedder
# (Gemini alternative: LLM_MODEL = "gemini-2.5-flash" — needs GOOGLE_API_KEY in .env)
BGE_MODEL_NAME = "BAAI/bge-base-en-v1.5"
PREVIEW = 62  # max characters of passage text shown next to each hit


## 2 · Load — corpus + questions from the fresh parquet files

**WHAT:** `load_passages` pulls the first `n` passages (text + ids) from
`passages.parquet`; `load_questions` pulls specific rows by id from
`test.parquet`; `preview` flattens a passage for one-line printing.

**WHY:** Identical helpers to track 04 keep the labs directly comparable —
the only thing that changes is the retriever wrapper.


In [4]:
def load_passages(path: Path, n: int) -> tuple[list[str], list[int]]:
    """Return (passage_texts, passage_ids) for the first ``n`` passages."""
    df = pd.read_parquet(path)
    subset = df.head(n)
    return subset["passage"].tolist(), subset.index.tolist()


def load_questions(path: Path, ids: list[int]) -> list[tuple[int, str]]:
    """Return [(question_id, question_text)] for the requested test rows."""
    df = pd.read_parquet(path)
    rows = df.loc[ids]
    return [(int(idx), row["question"]) for idx, row in rows.iterrows()]


def preview(text: str, limit: int = PREVIEW) -> str:
    """Flatten a passage for one-line printing."""
    flat = text.replace("\n", " ")
    return flat[:limit] + ("..." if len(flat) > limit else "")


## 3 · Experiment — raw vs rewritten retrieval for the same questions

**WHAT:** `run_experiment` embeds the 100-passage subset once (batched BGE),
builds the FAISS store, then for each question runs the plain
`SimilarityRetriever` AND the `QueryRewriteRetriever` over the same store —
recording the raw top-k, the LLM's rewritten query, its wall time, and the
rewritten top-k.

**WHY:** Both paths share one index, so any difference in the top-1 is
caused by the rewrite alone. The rewritten query is captured separately
(one extra `_rewrite` call) purely for the demo — the retrieval itself makes
exactly one LLM call per question.


In [5]:
def run_experiment() -> dict:
    passage_texts, passage_ids = load_passages(PASSAGES_PATH, N_PASSAGES)
    questions = load_questions(TEST_PATH, QUESTION_IDS)

    # --- Embed locally (BGE) and index in-memory ---------------------------
    embedder = BGEEmbedding(model_name=BGE_MODEL_NAME)
    t0 = time.perf_counter()
    passage_vecs = embedder.embed_documents(passage_texts)
    embed_s = time.perf_counter() - t0

    chunks = [
        Document(page_content=t, metadata={"id": pid})
        for t, pid in zip(passage_texts, passage_ids)
    ]
    store = FAISSVectorStore(embedding=embedder)
    t0 = time.perf_counter()
    store.add(chunks, embeddings=passage_vecs)
    index_s = time.perf_counter() - t0

    # --- The two retrievers over the SAME store -----------------------------
    raw_retriever = SimilarityRetriever(store, top_k=TOP_K)
    rewrite_llm = GroqLLM(model=LLM_MODEL)
    rewrite_retriever = QueryRewriteRetriever(rewrite_llm, raw_retriever, top_k=TOP_K)

    # --- Per question: raw retrieval + the rewritten query + its retrieval --
    results = []
    for qid, qtext in questions:
        raw_docs = raw_retriever.retrieve(qtext)
        t0 = time.perf_counter()
        rewritten = rewrite_retriever._rewrite(qtext)
        rewrite_s = time.perf_counter() - t0
        rewritten_docs = rewrite_retriever.retrieve(qtext)
        results.append(
            {
                "qid": qid,
                "question": qtext,
                "rewritten": rewritten,
                "rewrite_s": rewrite_s,
                "raw_docs": raw_docs,
                "rewritten_docs": rewritten_docs,
            }
        )

    return {
        "passage_texts": passage_texts,
        "passage_ids": passage_ids,
        "questions": questions,
        "indexed": len(passage_texts),
        "embed_s": embed_s,
        "index_s": index_s,
        "results": results,
    }


## 4 · Run — execute the experiment

**WHAT:** Calls `run_experiment()` — one embed, one index build, all
retrievals (plus the Groq transformation calls) — and keeps the artifact
dict as `exp`.

**WHY:** Everything after this cell (the demo and the verification gate)
reads from this single `exp`, so the printed numbers and the verified
numbers are guaranteed to come from the same run. The LLM calls happen here,
once — the gate cell never re-burns them.


In [6]:
exp = run_experiment()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## 5 · Demo — read the artifact

**WHAT:** `print_demo` prints the deterministic corpus summary, then per
question: the raw question, the LLM's rewritten query, and the top-1 passage
of both the raw and the rewritten path.

**WHY:** Read the rewritten line like a search engineer: it is the query the
*retriever* actually saw — standalone, specific, free of pronouns. When the
rewritten top-1 differs from the raw one, you are watching the technique
change retrieval; when it matches, the rewrite confirmed the raw query was
already good.


In [7]:
def print_demo(exp: dict) -> None:
    print("=" * 66)
    print("Lab 01 — Query rewrite: fix the query before you embed it")
    print(f"{BGE_MODEL_NAME} (local) -> FAISS top-{TOP_K} -> {LLM_MODEL} rewriter")
    print("=" * 66)

    print(f"\n[1] Corpus (deterministic subset, no randomness):")
    print(f"    {exp['indexed']} passages (first {N_PASSAGES} of 3200, ids {exp['passage_ids'][0]}..{exp['passage_ids'][-1]})")
    print(f"    embedded in {exp['embed_s']:.2f}s (dim 768), indexed in {exp['index_s']:.3f}s")

    print(f"\n[2] Raw vs rewritten (per question):")
    for r in exp["results"]:
        print(f'\n    Q[{r["qid"]}] "{r["question"]}"')
        print(f"      rewritten in {r['rewrite_s']:.1f}s: {r['rewritten']!r}")
        print(f"      raw      top-1: {preview(r['raw_docs'][0].page_content)}")
        print(f"      rewritten top-1: {preview(r['rewritten_docs'][0].page_content)}")

    print("\n[3] Takeaway")
    print("    Query rewrite is a retriever-layer fix: rewrite the question")
    print("    into a standalone, specific search query, THEN embed it. The")
    print("    store and the embedding model never change — only the text")
    print("    that reaches them. It costs one cheap LLM call per question")
    print("    and pays for itself whenever users type pronouns, ellipses,")
    print("    or conversational phrasing instead of search queries.")


In [8]:
print_demo(exp)


Lab 01 — Query rewrite: fix the query before you embed it
BAAI/bge-base-en-v1.5 (local) -> FAISS top-3 -> llama-3.3-70b-versatile rewriter

[1] Corpus (deterministic subset, no randomness):
    100 passages (first 100 of 3200, ids 0..99)
    embedded in 14.85s (dim 768), indexed in 0.044s

[2] Raw vs rewritten (per question):

    Q[1606] "Is Uruguay's capital Montevideo?"
      rewritten in 0.3s: 'Is Montevideo the capital of Uruguay?'
      raw      top-1: Montevideo, Uruguay's capital.
      rewritten top-1: Montevideo, Uruguay's capital.

    Q[1610] "Who founded Montevideo?"
      rewritten in 0.2s: 'Who founded Montevideo Uruguay'
      raw      top-1: Uruguay's capital, Montevideo, was founded by the Spanish in t...
      rewritten top-1: Uruguay's capital, Montevideo, was founded by the Spanish in t...

    Q[1626] "Did Uruguay host the first ever World Cup?"
      rewritten in 0.2s: 'Uruguay host first ever FIFA World Cup'
      raw      top-1: The main sport in Uruguay is foo

## 6 · Verification gate — the same checks the .py runs

**WHAT:** Runs the exact `verify_gate`: exactly `N_PASSAGES` indexed, every
question returning `TOP_K` hits on both paths, every rewrite non-empty and
*different* from the raw question, and the content checks — the rewritten
top-3 must still carry the answer's keyword (montevideo / spanish / 1930).

**WHY:** `python 01-rewrite.py --verify` must print 12/12 PASS; this cell
proves the notebook reproduces the verified `.py` exactly. The keyword checks
are pinned to *retrieval outcomes*, not exact LLM wording, so the gate stays
stable across runs.


In [9]:
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []

    # Structural properties (no LLM involved).
    checks.append((f"exactly {N_PASSAGES} passages indexed", exp["indexed"] == N_PASSAGES))
    checks.append(("each question returns TOP_K raw hits",
                   all(len(r["raw_docs"]) == TOP_K for r in exp["results"])))
    checks.append(("each question returns TOP_K rewritten hits",
                   all(len(r["rewritten_docs"]) == TOP_K for r in exp["results"])))

    # The rewrite must actually produce a standalone question — non-empty
    # and different from the original (the LLM resolves the vague phrasing).
    for r in exp["results"]:
        tag = f"Q{r['qid']}"
        checks.append((f"{tag} rewrite is non-empty",
                       bool(r["rewritten"].strip())))
        checks.append((f"{tag} rewrite differs from the raw question",
                       r["rewritten"].strip() != r["question"].strip()))

    # Content checks: the rewritten path must still surface the answer's
    # keyword. Q1606 "Is Uruguay's capital Montevideo?" -> Montevideo;
    # Q1610 "Who founded Montevideo?" -> the Spanish; Q1626 "Did Uruguay
    # host the first ever World Cup?" -> 1930.
    for r in exp["results"]:
        tag = f"Q{r['qid']}"
        joined = " ".join(d.page_content for d in r["rewritten_docs"]).lower()
        if r["qid"] == 1606:
            kw = "montevideo"
        elif r["qid"] == 1610:
            kw = "spanish"
        else:  # 1626
            kw = "1930"
        checks.append((f"{tag} rewritten top-{TOP_K} retains '{kw}'", kw in joined))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


In [10]:
verify_gate(exp)


verification gate:
  [PASS] exactly 100 passages indexed
  [PASS] each question returns TOP_K raw hits
  [PASS] each question returns TOP_K rewritten hits
  [PASS] Q1606 rewrite is non-empty
  [PASS] Q1606 rewrite differs from the raw question
  [PASS] Q1610 rewrite is non-empty
  [PASS] Q1610 rewrite differs from the raw question
  [PASS] Q1626 rewrite is non-empty
  [PASS] Q1626 rewrite differs from the raw question
  [PASS] Q1606 rewritten top-3 retains 'montevideo'
  [PASS] Q1610 rewritten top-3 retains 'spanish'
  [PASS] Q1626 rewritten top-3 retains '1930'


0